In [ ]:
Тема диплома: Сравнительный анализ эффективности современных методов токенизации и векторизации в задачах классификации и кластеризации текстов
Краткое напоминание (для меня):
Токенизация - как режем строки (по пробелам, знакам препинания, специфические способы и тд), те способо нарезки
Векторизация - математическая модель. Возвращает векторы(наборы чисел), на которых учится классификатор, те способ оцифровки
Классификация - способ принятия решения на основе векторов

В этом файле использованы следующие методы векторзации:
Binary, Bag of Words, TF-IDF Standart, TF-IDF Bigrams, Word2Vec, Doc2Vec, Bert
Модель - Логистическая регрессия

В результате победил метод TF-IDF Standard
	Method	               Accuracy	Time (sec)
2	TF-IDF Standard	        0.8647	6.90
0	Binary (Presence)	    0.8613	7.12
3	TF-IDF Bigrams	        0.8576	13.83
1	Bag of Words (Counts)	0.8520	11.57
6	BERT (all-MiniLM-L6-v2)	0.8138	925.73
5	Doc2Vec	                0.6631	241.03
4	Word2Vec (Average)	    0.5117	30.28

Эта табличка сохраняется в /results/tables/vectorization_comparison.csv

In [18]:
import pandas as pd
import time
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score
from gensim.models import Word2Vec
import numpy as np
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from sentence_transformers import SentenceTransformer
import os

# Загружаем данные один раз
data = fetch_20newsgroups(subset='all')
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=42)

In [21]:
# Список методов для сравнения
methods = {
    "Binary (Presence)": CountVectorizer(binary=True, stop_words='english', max_features=5000),
    "Bag of Words (Counts)": CountVectorizer(binary=False, stop_words='english', max_features=5000),
    "TF-IDF Standard": TfidfVectorizer(stop_words='english', max_features=5000),
    "TF-IDF Bigrams": TfidfVectorizer(stop_words='english', max_features=5000, ngram_range=(1, 2))
}

comparison_results = []

for name, vectorizer in methods.items():
    print(f"Testing method: {name}...")
    
    # Засекаем время для анализа производительности
    start_time = time.time()
    
    # 1. Векторизация
    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec = vectorizer.transform(X_test)
    
    # 2. Обучение модели
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train_vec, y_train)
    
    # 3. Оценка
    y_pred = model.predict(X_test_vec)
    acc = accuracy_score(y_test, y_pred)
    
    end_time = time.time()
    dur = round(end_time - start_time, 2)
    print(f"Method {name} completed in {dur}")
    print(f"Accuracy for method {name} = {acc}")
    # Сохраняем результаты
    comparison_results.append({
        "Method": name,
        "Accuracy": round(acc, 4),
        "Time (sec)": dur
    })

Testing method: Binary (Presence)...
Method Binary (Presence) completed in 7.12
Accuracy for method Binary (Presence) = 0.8612732095490716
Testing method: Bag of Words (Counts)...
Method Bag of Words (Counts) completed in 11.57
Accuracy for method Bag of Words (Counts) = 0.8519893899204244
Testing method: TF-IDF Standard...
Method TF-IDF Standard completed in 6.9
Accuracy for method TF-IDF Standard = 0.8647214854111406
Testing method: TF-IDF Bigrams...
Method TF-IDF Bigrams completed in 13.83
Accuracy for method TF-IDF Bigrams = 0.8575596816976128


Word2Vec

In [22]:
# 0. Начинаем отсчет времени
start_w2v = time.time()

# 1. Простая токенизация для Word2Vec (делим строки на списки слов)
tokenized_train = [text.lower().split() for text in X_train]
tokenized_test = [text.lower().split() for text in X_test]

# 2. Обучение модели Word2Vec
# size=100 (размер вектора), window=5 (контекст), min_count=2 (игнорим редкие слова)
w2v_model = Word2Vec(sentences=tokenized_train, vector_size=100, window=5, min_count=2, workers=4)

# 3. Функция для превращения текста в один средний вектор
def get_text_vector(tokens, model):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

# Превращаем все тексты в наборы чисел
X_train_w2v = np.array([get_text_vector(t, w2v_model) for t in tokenized_train])
X_test_w2v = np.array([get_text_vector(t, w2v_model) for t in tokenized_test])

# 4. Обучение логистической регрессии на этих векторах
model_w2v = LogisticRegression(max_iter=1000)
model_w2v.fit(X_train_w2v, y_train)

# 5. Результат
acc_w2v = accuracy_score(y_test, model_w2v.predict(X_test_w2v))

# 6. Фиксируем окончание
end_w2v = time.time()
dur_w2v = round(end_w2v - start_w2v, 2)

print(f"Word2Vec отработал за {dur_w2v}")
print(f"Word2Vec Accuracy: {acc_w2v}")
comparison_results.append({"Method": "Word2Vec (Average)", 
                          "Accuracy": round(acc_w2v, 4), 
                          "Time (sec)": dur_w2v}) 

Word2Vec отработал за 30.28
Word2Vec Accuracy: 0.5116710875331565


Doc2Vec

In [23]:
# 0. Начинаем отсчет
start_d2v = time.time()

# 1. Подготовка данных (нужны помеченные документы)
tagged_data = [TaggedDocument(words=text.lower().split(), tags=[str(i)]) for i, text in enumerate(X_train)]

# 2. Обучение Doc2Vec
d2v_model = Doc2Vec(vector_size=100, window=5, min_count=2, workers=4, epochs=20)
d2v_model.build_vocab(tagged_data)
d2v_model.train(tagged_data, total_examples=d2v_model.corpus_count, epochs=d2v_model.epochs)

# 3. Получение векторов
X_train_d2v = np.array([d2v_model.infer_vector(text.lower().split()) for text in X_train])
X_test_d2v = np.array([d2v_model.infer_vector(text.lower().split()) for text in X_test])

# 4. Обучение классификатора
model_d2v = LogisticRegression(max_iter=1000)
model_d2v.fit(X_train_d2v, y_train)
acc_d2v = accuracy_score(y_test, model_d2v.predict(X_test_d2v))

# 5. Фиксируем окончание
end_d2v = time.time()
dur_d2v = round(end_d2v - start_d2v, 2)

print(f"Doc2Vec отработал за {dur_d2v}")
print(f"Doc2Vec Accuracy: {acc_d2v}")

comparison_results.append({"Method": "Doc2Vec", 
                          "Accuracy": round(acc_d2v, 4), 
                          "Time (sec)": dur_d2v}) 

Doc2Vec отработал за 241.03
Doc2Vec Accuracy: 0.6631299734748011


BERT
Долго работает, не запускать без необходимости!

In [24]:
# 1. Загружаем модель
model_bert_strong = SentenceTransformer('all-MiniLM-L6-v2')

print("Начинаю полный цикл BERT (кодирование + обучение)...")

# СТАРТ ЗАМЕРА
full_start_time = time.time()

# 2. Кодирование
X_train_bert_full = model_bert_strong.encode(X_train, show_progress_bar=True, batch_size=32)
X_test_bert_full = model_bert_strong.encode(X_test, show_progress_bar=True, batch_size=32)

# 3. Обучение логистической регрессии
clf_bert = LogisticRegression(max_iter=1000)
clf_bert.fit(X_train_bert_full, y_train)

# КОНЕЦ ЗАМЕРА
full_end_time = time.time()

# 4. Проверяем результат
y_pred_bert = clf_bert.predict(X_test_bert_full)
final_acc_bert = accuracy_score(y_test, y_pred_bert)

full_dur_bert = round(full_end_time-full_start_time, 2) 

print(f"Кодирование и обучение завершены за {full_dur_bert}")
print(f"Финальная точность BERT: {final_acc_bert:.4f}")

# 5. Добавляем в итоговый список
comparison_results.append({
    "Method": "BERT (all-MiniLM-L6-v2)", 
    "Accuracy": round(final_acc_bert, 4), 
    "Time (sec)": full_dur_bert
})

Loading weights: 100%|█| 10
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Начинаю полный цикл BERT (кодирование + обучение)...


Batches: 100%|█| 472/472 [0
Batches: 100%|█| 118/118 [0


Кодирование и обучение завершены за 925.73
Финальная точность BERT: 0.8138


In [25]:
# Сохраняем в папку data_old/processed/
os.makedirs('../data_old/processed/', exist_ok=True)
os.makedirs("../results/tables", exist_ok=True)
np.save('../data_old/processed/train_bert_vectors.npy', X_train_bert_full)
np.save('../data_old/processed/test_bert_vectors.npy', X_test_bert_full)

In [32]:
# Создаем таблицу
df_results = pd.DataFrame(comparison_results)

# Сортируем по точности, чтобы лучший метод был сверху
df_results = df_results.sort_values(by=["Accuracy"], ascending=False)

# Сохраняем в папку results/tables
df_results.to_csv('../results/tables/vectorization_comparison.csv', index=False)

# Выводим в ноутбук
df_results

,Method,Accuracy,Time (sec)
2,TF-IDF Standard,0.8647,6.90
0,Binary (Presence),0.8613,7.12
3,TF-IDF Bigrams,0.8576,13.83
1,Bag of Words (Counts),0.8520,11.57
6,BERT (all-MiniLM-L6-v2),0.8138,925.73
5,Doc2Vec,0.6631,241.03
4,Word2Vec (Average),0.5117,30.28


The end